# SCOTUS Corpus Analysis — Class/Identity Spectral Decomposition (1873–2018)

Standalone analysis of 55 SCOTUS opinion PDFs from the Internet Archive collection.
Runs independently and exports `Paper/data/scotus_spectral_results.json` for
consumption by `Paper/scripts/eq40_45_interference_engine.ipynb`.

**Pre-requisite**: Run `python3 Paper/scripts/scotus_text_extract.py` and then
`make data-refresh` to generate `Paper/data/scotus_keyword_counts.csv`.

## Three analyses

- **Analysis 1**: Class/Identity keyword ratio trend over time (1873–2018)
- **Analysis 2**: Per-axis Lomb-Scargle periodogram on non-uniform sample
- **Analysis 3**: Majority vs. dissent framing divergence

## Equations targeted

- `eq:41` — per-axis identity-band frequencies (distinct f_k per axis)
- `eq:40` — class-band suppression in institutional language

## Inputs
- `Paper/data/scotus_keyword_counts.csv`

## Outputs
- `Paper/figures/spectral/scotus_class_identity_ratio.pdf`
- `Paper/figures/spectral/scotus_lomb_scargle.pdf`
- `Paper/figures/spectral/scotus_majority_dissent.pdf`
- `Paper/data/scotus_spectral_results.json` (consumed by main notebook)

**Note**: 55 cases is a small corpus for spectral analysis. Lomb-Scargle handles
non-uniform sampling but frequency resolution is limited. The value is the
145-year time span enabling detection of multi-decade cycles.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import lombscargle
from scipy.stats import linregress

warnings.filterwarnings('ignore', category=RuntimeWarning)

try:
    _here = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    if (_cwd / 'data').exists():
        _here = _cwd
    elif (_cwd / 'Paper' / 'data').exists():
        _here = _cwd / 'Paper' / 'scripts'
    else:
        _here = _cwd

DATA = _here / '..' / 'data'
FIGS = _here / '..' / 'figures' / 'spectral'
FIGS.mkdir(parents=True, exist_ok=True)

SCOTUS_CSV = DATA / 'scotus_keyword_counts.csv'
RESULTS_JSON = DATA / 'scotus_spectral_results.json'

print(f'Data dir:    {DATA.resolve()}')
print(f'Figures dir: {FIGS.resolve()}')
print(f'SCOTUS CSV:  {SCOTUS_CSV.exists()}')

Data dir:    /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/data
Figures dir: /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/figures/spectral
SCOTUS CSV:  True


In [2]:
DATA_AVAILABLE = SCOTUS_CSV.exists()

if DATA_AVAILABLE:
    scotus = pd.read_csv(SCOTUS_CSV, comment='#')
    # Validate expected columns
    required_cols = ['case_name', 'year', 'total_words', 'class_per_1k',
                     'race_per_1k', 'gender_per_1k', 'religion_per_1k',
                     'sexuality_per_1k', 'class_share', 'identity_share', 'is_dissent']
    missing = [c for c in required_cols if c not in scotus.columns]
    if missing:
        print(f'WARNING: missing columns: {missing}')
        DATA_AVAILABLE = False
    else:
        print(f'Loaded: {len(scotus)} rows, {scotus["case_name"].nunique()} unique cases')
        print(f'Year range: {scotus["year"].min()} – {scotus["year"].max()}')
        assert scotus['year'].between(1870, 2020).all(), 'Year out of expected range'
        assert scotus['total_words'].gt(0).all(), 'Zero word count detected'
else:
    print('DATA NOT AVAILABLE: scotus_keyword_counts.csv not found.')
    print('Run:  python3 Paper/scripts/scotus_text_extract.py')
    print('Then: make data-refresh')
    print('Notebook will run in stub mode, exporting placeholder results.')

Loaded: 59 rows, 57 unique cases
Year range: 1873 – 2018


## Analysis 1: Class/Identity Keyword Ratio Over Time

**Prediction (eq:40)**: Class language share declines after the mid-20th century
while identity language rises — mirroring the Congressional Record and Google
Trends patterns, but in judicial language.

Operationalization:
```
class_share = class_per_1k / (class_per_1k + identity_per_1k)
```
where `identity_per_1k = race_per_1k + gender_per_1k + religion_per_1k + sexuality_per_1k`

In [3]:
if DATA_AVAILABLE:
    # Majority opinions only for the primary trend analysis
    majority = scotus[~scotus['is_dissent']].copy()
    majority = majority.sort_values('year').reset_index(drop=True)

    # Linear trend fit for class_share
    valid = majority.dropna(subset=['class_share', 'year'])
    slope, intercept, r_value, p_value, se = linregress(valid['year'], valid['class_share'])
    trend_x = np.linspace(valid['year'].min(), valid['year'].max(), 200)
    trend_y = slope * trend_x + intercept

    # Era means: pre-1950 vs post-1950
    pre1950  = valid[valid['year'] <  1950]['class_share'].mean()
    post1950 = valid[valid['year'] >= 1950]['class_share'].mean()

    print('=== Analysis 1: Class/Identity Ratio in SCOTUS Majority Opinions ===')
    print(f'  Pre-1950 mean class_share:  {pre1950:.3f}')
    print(f'  Post-1950 mean class_share: {post1950:.3f}')
    print(f'  Change:                     {post1950 - pre1950:+.3f}')
    print(f'  OLS slope: {slope:.5f}/yr   R²={r_value**2:.3f}   p={p_value:.4f}')

    a1_results = {
        'pre_1950_class_share_mean': round(float(pre1950), 4),
        'post_1950_class_share_mean': round(float(post1950), 4),
        'class_share_slope_per_yr': round(float(slope), 6),
        'class_share_r2': round(float(r_value**2), 4),
        'class_share_p': round(float(p_value), 4),
    }
else:
    print('STUB MODE — Analysis 1 skipped (data unavailable).')
    a1_results = {
        'pre_1950_class_share_mean': None,
        'post_1950_class_share_mean': None,
        'class_share_slope_per_yr': None,
        'class_share_r2': None,
        'class_share_p': None,
        'stub': True,
    }

=== Analysis 1: Class/Identity Ratio in SCOTUS Majority Opinions ===
  Pre-1950 mean class_share:  0.509
  Post-1950 mean class_share: 0.366
  Change:                     -0.144
  OLS slope: -0.00161/yr   R²=0.043   p=0.1243


In [4]:
if DATA_AVAILABLE:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.scatter(majority['year'], majority['class_share'],
               s=40, alpha=0.7, color='steelblue', zorder=3, label='Majority opinion')
    ax.plot(trend_x, trend_y, '--', color='firebrick', lw=1.8,
            label=f'OLS trend (slope={slope:.4f}/yr, R²={r_value**2:.2f})')
    ax.axvline(1950, color='gray', lw=1.0, ls=':', alpha=0.8)
    ax.text(1952, ax.get_ylim()[0] + 0.02, '1950', fontsize=8, color='gray')
    ax.set_xlabel('Year')
    ax.set_ylabel('Class-language share')
    ax.set_title('SCOTUS Majority Opinion: Class vs. Identity Language Share (1873–2018)')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    fig.tight_layout()
    out_a1 = FIGS / 'scotus_class_identity_ratio.pdf'
    fig.savefig(out_a1, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_a1}')

Saved: /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/scripts/../figures/spectral/scotus_class_identity_ratio.pdf


## Analysis 2: Per-Axis Lomb-Scargle Periodogram

The 55 cases are non-uniformly spaced across 145 years — standard FFT cannot
be used. We use `scipy.signal.lombscargle` (Lomb-Scargle periodogram, designed
for non-uniformly sampled time series; Scargle 1982).

**Key test**: eq:41 predicts each axis operates at a distinct frequency f_k.
We report whether the per-axis dominant periods are distinct within the
identity band.

**Methodological note**: 55 cases is a small corpus. Frequency resolution is
limited. Results should be treated as exploratory and cross-validated with
GDELT and Congressional Record data.

In [5]:
IDENTITY_AXES = ['race', 'gender', 'religion', 'sexuality']
AXIS_COLORS   = {'race': 'firebrick', 'gender': 'steelblue',
                 'religion': 'darkorange', 'sexuality': 'mediumseagreen'}

# Period range for Lomb-Scargle: 3 to 50 years (in years)
PERIOD_MIN_YR = 3.0
PERIOD_MAX_YR = 50.0
N_FREQ = 500

if DATA_AVAILABLE:
    # Use majority opinions only; take one row per case (average if split)
    maj_per_case = (scotus[~scotus['is_dissent']]
                    .groupby(['case_name', 'year'])
                    .mean(numeric_only=True)
                    .reset_index())
    maj_per_case = maj_per_case.sort_values('year').reset_index(drop=True)

    # Time vector in years (fractional year as float)
    t_yr = maj_per_case['year'].values.astype(float)

    # Angular frequency grid (rad/year)
    freq_grid = np.linspace(1.0 / PERIOD_MAX_YR, 1.0 / PERIOD_MIN_YR, N_FREQ)  # cycles/yr
    omega_grid = 2 * np.pi * freq_grid  # rad/yr

    ls_results: dict[str, dict] = {}
    for axis in IDENTITY_AXES:
        col = f'{axis}_per_1k'
        if col not in maj_per_case.columns:
            print(f'  Missing column {col} — skipping axis {axis}')
            continue
        signal = maj_per_case[col].values.astype(float)
        # Lomb-Scargle requires zero-mean signal
        signal -= signal.mean()
        pgram = lombscargle(t_yr, signal, omega_grid, normalize=True)
        peak_idx = np.argmax(pgram)
        dominant_period = 1.0 / freq_grid[peak_idx]
        ls_results[axis] = {
            'dominant_period_yr': round(float(dominant_period), 2),
            'peak_power': round(float(pgram[peak_idx]), 4),
            'pgram': pgram.tolist(),
        }
        print(f'  {axis:12s}: dominant period = {dominant_period:.1f} yr  '
              f'(peak power = {pgram[peak_idx]:.3f})')

    # --- Boundary artifact check ---
    # If an axis's dominant period == PERIOD_MAX_YR (or within 1 yr), it is a
    # boundary artifact: the periodogram power peaks at the edge of the search
    # grid, meaning the true period may be longer or the signal too sparse to
    # resolve.  Exclude boundary-artifact axes from the distinctness calculation.
    BOUNDARY_TOL = 1.0  # yr
    boundary_axes = []
    for ax, res in ls_results.items():
        if abs(res['dominant_period_yr'] - PERIOD_MAX_YR) <= BOUNDARY_TOL:
            boundary_axes.append(ax)
            res['boundary_artifact'] = True
            print(f'  ⚠️  {ax}: dominant period {res["dominant_period_yr"]:.1f} yr '
                  f'== search boundary ({PERIOD_MAX_YR} yr) — flagged as boundary artifact. '
                  'True period may exceed search range; exclude from distinctness test.')
        else:
            res['boundary_artifact'] = False

    # Distinctness based only on non-artifact axes
    valid_periods = [v['dominant_period_yr'] for k, v in ls_results.items()
                     if not v['boundary_artifact']]
    period_range_valid = max(valid_periods) - min(valid_periods) if len(valid_periods) > 1 else 0
    distinct = period_range_valid > 3.0 and len(valid_periods) >= 2

    # Full range (for reporting, including artifacts)
    periods_all = [v['dominant_period_yr'] for v in ls_results.values()]
    period_range_all = max(periods_all) - min(periods_all) if periods_all else 0

    print(f'\n  Valid (non-artifact) axes: {[k for k in ls_results if not ls_results[k]["boundary_artifact"]]}')
    print(f'  Period range (valid axes only): {period_range_valid:.1f} yr')
    print(f'  Periods DISTINCT (>3 yr, valid only): {distinct}')
    if boundary_axes:
        print(f'  Boundary-artifact axes excluded from distinctness: {boundary_axes}')
    if distinct:
        print('  → CONSISTENT with eq:41 (each axis at distinct f_k, non-artifact axes)')
    elif len(valid_periods) < 2:
        print('  → INDETERMINATE — fewer than 2 non-artifact axes; cannot assess distinctness')
    else:
        print('  → INCONSISTENT with eq:41 — valid periods cluster')

    a2_results = {
        'per_axis_dominant_periods': {k: v['dominant_period_yr'] for k, v in ls_results.items()},
        'boundary_artifact_axes': boundary_axes,
        'period_range_yr_all_axes': round(float(period_range_all), 2),
        'period_range_yr_valid_axes': round(float(period_range_valid), 2),
        'periods_distinct_3yr': bool(distinct),
        'n_valid_axes': len(valid_periods),
        'corpus_size_cases': int(maj_per_case.shape[0]),
        'year_span': int(t_yr.max() - t_yr.min()),
        'freq_grid_cycles_per_yr': freq_grid.tolist(),
        'per_axis_pgrams': {k: v['pgram'] for k, v in ls_results.items()},
        'per_axis_boundary_artifact': {k: v['boundary_artifact'] for k, v in ls_results.items()},
    }
else:
    print('STUB MODE — Analysis 2 skipped.')
    a2_results = {
        'per_axis_dominant_periods': {ax: None for ax in IDENTITY_AXES},
        'period_range_yr': None,
        'periods_distinct_3yr': None,
        'corpus_size_cases': 55,
        'year_span': 145,
        'stub': True,
    }

  race        : dominant period = 3.6 yr  (peak power = 0.122)
  gender      : dominant period = 6.2 yr  (peak power = 0.167)
  religion    : dominant period = 8.5 yr  (peak power = 0.144)
  sexuality   : dominant period = 50.0 yr  (peak power = 0.167)
  ⚠️  sexuality: dominant period 50.0 yr == search boundary (50.0 yr) — flagged as boundary artifact. True period may exceed search range; exclude from distinctness test.

  Valid (non-artifact) axes: ['race', 'gender', 'religion']
  Period range (valid axes only): 4.9 yr
  Periods DISTINCT (>3 yr, valid only): True
  Boundary-artifact axes excluded from distinctness: ['sexuality']
  → CONSISTENT with eq:41 (each axis at distinct f_k, non-artifact axes)


In [6]:
if DATA_AVAILABLE and ls_results:
    period_grid_yr = 1.0 / freq_grid  # period in years

    fig, ax = plt.subplots(figsize=(11, 4.5))
    for axis, res in ls_results.items():
        pgram = np.array(res['pgram'])
        ax.plot(period_grid_yr, pgram, lw=1.6, color=AXIS_COLORS[axis],
                label=f'{axis.capitalize()} (dom. {res["dominant_period_yr"]:.0f} yr)', alpha=0.85)
        dp = res['dominant_period_yr']
        ax.axvline(dp, color=AXIS_COLORS[axis], lw=0.8, ls='--', alpha=0.6)

    ax.set_xlabel('Period (years)')
    ax.set_ylabel('Normalised Lomb-Scargle power')
    ax.set_title('SCOTUS Corpus: Per-Axis Lomb-Scargle Periodogram (1873–2018)')
    ax.legend(fontsize=9)
    ax.set_xlim(PERIOD_MIN_YR, PERIOD_MAX_YR)
    ax.text(0.98, 0.97,
            f'N={int(a2_results["corpus_size_cases"])} cases | '
            f'{int(a2_results["year_span"])}-yr span | Scargle (1982)',
            transform=ax.transAxes, ha='right', va='top', fontsize=8, color='gray')
    fig.tight_layout()
    out_a2 = FIGS / 'scotus_lomb_scargle.pdf'
    fig.savefig(out_a2, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_a2}')

Saved: /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/scripts/../figures/spectral/scotus_lomb_scargle.pdf


## Analysis 3: Majority vs. Dissent Framing Divergence

**Prediction**: Dissents use more class language than majorities — dissents are
where suppressed framings survive (e.g., Harlan in *Plessy*, *Civil Rights Cases*).

Operationalization: For each case that has both majority and dissent sections,
compute the divergence ratio:
```
divergence = class_per_1k(dissent) / class_per_1k(majority)
```
Cases with divergence > 1.5 are labeled as high-divergence (contested framings).

In [7]:
if DATA_AVAILABLE:
    # Pivot to get majority and dissent side-by-side per case
    maj_d  = scotus[~scotus['is_dissent']][['case_name', 'year', 'class_per_1k']].rename(
        columns={'class_per_1k': 'maj_class'})
    dis_d  = scotus[ scotus['is_dissent']][['case_name', 'year', 'class_per_1k']].rename(
        columns={'class_per_1k': 'dis_class'})
    paired = maj_d.merge(dis_d, on=['case_name', 'year'], how='inner')


    # --- Underpowered warning ---
    MIN_PAIRS_FOR_INFERENCE = 10
    if len(paired) < MIN_PAIRS_FOR_INFERENCE:
        import warnings
        warnings.warn(
            f"Analysis 3 is UNDERPOWERED: only {len(paired)} paired cases found "
            f"(minimum {MIN_PAIRS_FOR_INFERENCE} required for directional inference). "
            "Results are illustrative only and must NOT be cited as confirmatory evidence.",
            stacklevel=2,
        )
        print(f"\n⚠️  WARNING: N={len(paired)} paired cases — analysis is underpowered.")
        print("   Directional inference requires N ≥ 10. Results are exploratory/illustrative only.")
        print("   Note: ratio < 1.0 means majorities use MORE class language than dissents")
        print("   (opposite to prediction). Cannot conclude direction from N=2.\n")

    eps = 1e-6  # avoid division by zero
    paired['divergence_ratio'] = paired['dis_class'] / (paired['maj_class'] + eps)
    paired = paired.sort_values('divergence_ratio', ascending=False).reset_index(drop=True)

    high_div = paired[paired['divergence_ratio'] > 1.5]
    mean_maj = paired['maj_class'].mean()
    mean_dis = paired['dis_class'].mean()

    print('=== Analysis 3: Majority vs. Dissent Class-Language Divergence ===')
    print(f'  Cases with both majority & dissent: {len(paired)}')
    print(f'  Mean class_per_1k (majority):  {mean_maj:.3f}')
    print(f'  Mean class_per_1k (dissent):   {mean_dis:.3f}')
    print(f'  Dissent/majority ratio (mean): {mean_dis/(mean_maj+eps):.3f}')
    print(f'  High-divergence cases (>1.5x): {len(high_div)}')
    if not high_div.empty:
        print('  Top high-divergence cases:')
        for _, row in high_div.head(5).iterrows():
            print(f'    {row["case_name"]} ({int(row["year"])}): '
                  f'maj={row["maj_class"]:.2f}, dis={row["dis_class"]:.2f}, '
                  f'ratio={row["divergence_ratio"]:.2f}')

    a3_results = {
        'n_cases_with_dissent': int(len(paired)),
        'mean_class_per_1k_majority': round(float(mean_maj), 4),
        'mean_class_per_1k_dissent': round(float(mean_dis), 4),
        'dissent_majority_ratio': round(float(mean_dis / (mean_maj + eps)), 4),
        'n_high_divergence_cases': int(len(high_div)),
        'top_high_divergence_cases': high_div.head(5)[['case_name', 'year', 'divergence_ratio']].to_dict('records'),
        'underpowered': bool(len(paired) < MIN_PAIRS_FOR_INFERENCE),
        'note': ('UNDERPOWERED' if len(paired) < MIN_PAIRS_FOR_INFERENCE else 'adequate'),
        'direction_note': ('majority > dissent (opposite to prediction)' if mean_dis < mean_maj else 'dissent >= majority (consistent with prediction)'),
    }
else:
    print('STUB MODE — Analysis 3 skipped.')
    a3_results = {
        'n_cases_with_dissent': None,
        'mean_class_per_1k_majority': None,
        'mean_class_per_1k_dissent': None,
        'dissent_majority_ratio': None,
        'n_high_divergence_cases': None,
        'top_high_divergence_cases': [],
        'stub': True,
    }


⚠️  WARNING: N=2 paired cases — analysis is underpowered.
   Directional inference requires N ≥ 10. Results are exploratory/illustrative only.
   Note: ratio < 1.0 means majorities use MORE class language than dissents
   (opposite to prediction). Cannot conclude direction from N=2.

=== Analysis 3: Majority vs. Dissent Class-Language Divergence ===
  Cases with both majority & dissent: 2
  Mean class_per_1k (majority):  2.210
  Mean class_per_1k (dissent):   0.913
  Dissent/majority ratio (mean): 0.413
  High-divergence cases (>1.5x): 1
  Top high-divergence cases:
    Jones v. Alfred H. Mayer Co. (1968): maj=0.17, dis=0.26, ratio=1.55


/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3747: UserWarning: Analysis 3 is UNDERPOWERED: only 2 paired cases found (minimum 10 required for directional inference). Results are illustrative only and must NOT be cited as confirmatory evidence.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
if DATA_AVAILABLE and not paired.empty:
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter_kwargs = dict(alpha=0.65, s=50, zorder=3)

    # Colour high-divergence cases differently
    is_high = paired['divergence_ratio'] > 1.5
    ax.scatter(paired.loc[~is_high, 'maj_class'], paired.loc[~is_high, 'dis_class'],
               color='steelblue', label='Standard cases', **scatter_kwargs)
    ax.scatter(paired.loc[is_high, 'maj_class'], paired.loc[is_high, 'dis_class'],
               color='firebrick', label='High divergence (>1.5×)', **scatter_kwargs)

    # 45-degree parity line
    lim = max(paired['maj_class'].max(), paired['dis_class'].max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', lw=1.0, alpha=0.5, label='Parity')
    ax.plot([0, lim], [0, 1.5 * lim], ':', color='firebrick', lw=1.0, alpha=0.5,
            label='1.5× divergence threshold')

    # Label top 5 high-divergence cases
    for _, row in high_div.head(5).iterrows():
        ax.annotate(f"{row['case_name'].split(' v. ')[0]} ({int(row['year'])})",
                    xy=(row['maj_class'], row['dis_class']),
                    xytext=(4, 4), textcoords='offset points', fontsize=7, color='firebrick')

    ax.set_xlabel('Class-language freq. in Majority Opinion (per 1,000 words)')
    ax.set_ylabel('Class-language freq. in Dissent (per 1,000 words)')
    ax.set_title('SCOTUS: Majority vs. Dissent Class-Language Frequency (1873–2018)')
    ax.legend(fontsize=9)
    fig.tight_layout()
    out_a3 = FIGS / 'scotus_majority_dissent.pdf'
    fig.savefig(out_a3, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_a3}')

Saved: /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/scripts/../figures/spectral/scotus_majority_dissent.pdf


## Export Results JSON

Writes `Paper/data/scotus_spectral_results.json` for consumption by
`eq40_45_interference_engine.ipynb`.

In [9]:
results = {
    'source': 'scotus_corpus_analysis.ipynb',
    'corpus': {
        'n_pdfs': 55,
        'year_range': '1873–2018',
        'data_available': DATA_AVAILABLE,
    },
    'analysis_1_class_identity_ratio': a1_results,
    'analysis_2_lomb_scargle': {
        'per_axis_dominant_periods': a2_results.get('per_axis_dominant_periods', {}),
        'boundary_artifact_axes': a2_results.get('boundary_artifact_axes', []),
        'per_axis_boundary_artifact': a2_results.get('per_axis_boundary_artifact', {}),
        'period_range_yr_all_axes': a2_results.get('period_range_yr_all_axes'),
        'period_range_yr_valid_axes': a2_results.get('period_range_yr_valid_axes'),
        'periods_distinct_3yr': a2_results.get('periods_distinct_3yr'),
        'n_valid_axes': a2_results.get('n_valid_axes'),
        'corpus_size_cases': a2_results.get('corpus_size_cases', 55),
        'year_span': a2_results.get('year_span', 145),
        'stub': a2_results.get('stub', False),
    },
    'analysis_3_majority_dissent': a3_results,
    'figures': [
        'Paper/figures/spectral/scotus_class_identity_ratio.pdf',
        'Paper/figures/spectral/scotus_lomb_scargle.pdf',
        'Paper/figures/spectral/scotus_majority_dissent.pdf',
    ],
    'note': (
        '55-case corpus; Lomb-Scargle handles non-uniform sampling but '
        'frequency resolution is limited by corpus size. '
        'Results are exploratory. Cross-validate with GDELT and CR data.'
    ),
}

with RESULTS_JSON.open('w') as fh:
    json.dump(results, fh, indent=2)
print(f'Exported: {RESULTS_JSON}')

print('\n=== SCOTUS Corpus Analysis Complete ===')
print(f'  Data available: {DATA_AVAILABLE}')
if DATA_AVAILABLE:
    print(f'  Analysis 1 — class_share slope: {a1_results["class_share_slope_per_yr"]:.5f}/yr, R²={a1_results["class_share_r2"]:.3f}')
    print(f'  Analysis 2 — per-axis periods: {a2_results["per_axis_dominant_periods"]}')
    print(f'  Analysis 3 — dissent/majority class ratio: {a3_results["dissent_majority_ratio"]:.3f}')
else:
    print('  Stub results written. Run scotus_text_extract.py to enable full analysis.')

Exported: /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/scripts/../data/scotus_spectral_results.json

=== SCOTUS Corpus Analysis Complete ===
  Data available: True
  Analysis 1 — class_share slope: -0.00161/yr, R²=0.043
  Analysis 2 — per-axis periods: {'race': 3.59, 'gender': 6.18, 'religion': 8.52, 'sexuality': 50.0}
  Analysis 3 — dissent/majority class ratio: 0.413
